# Spark Architecture: Partitions, Jobs, Stages, and Tasks

This notebook demonstrates Spark's execution engine model using PySpark:
1. **Partitions:** How data is split for parallel processing. We will look at `repartition` (wide) vs. `coalesce` (narrow).
2. **Jobs:** Triggered by Spark actions (`count`, `show`).
3. **Stages:** Divided by shuffle boundaries (narrow vs. wide transformations).
4. **Tasks:** Individual units of execution matching partitions.

## Step 1: Initialize Spark Session

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder \
    .appName("Lab1_1-Spark-Architecture-Exploration") \
    .getOrCreate()

print("SparkSession successfully initialized!")

## Step 2: Partitions in Spark

Let's read the `ratings.csv` dataset from HDFS (make sure to upload and put it in HDFS as instructed in the README).

In [ ]:
ratings_path = "hdfs:///data/ratings.csv"
df = spark.read.csv(ratings_path, header=True, inferSchema=True)

# Check the initial partition count
initial_partitions = df.rdd.getNumPartitions()
print(f"Initial partition count: {initial_partitions}")

### Repartition vs. Coalesce
- `repartition(n)`: Performs a full shuffle to redistribute data evenly across `n` partitions (Wide transformation - starts a new Stage).
- `coalesce(n)`: Decreases partition count without a full shuffle (Narrow transformation - executes inside the same Stage).

In [ ]:
# Increase partitions to 4 (requires a full shuffle)
df_repartitioned = df.repartition(4)
print(f"Partitions after repartition(4): {df_repartitioned.rdd.getNumPartitions()}")

# Reduce partitions to 2 (coalesces partitions, no shuffle)
df_coalesced = df_repartitioned.coalesce(2)
print(f"Partitions after coalesce(2): {df_coalesced.rdd.getNumPartitions()}")

## Step 3: Spark Jobs (Triggered by Actions)

Spark uses **lazy evaluation**. Transformations are not run until an **Action** (such as `count`, `show`, `collect`, `save`) is executed. Each action starts a new **Job**.

In [ ]:
# Trigger Action 1: count() -> Triggers Job 0
total_records = df.count()
print(f"[Action 1] Total records: {total_records}")

# Trigger Action 2: show(5) -> Triggers Job 1
print("[Action 2] Displaying sample ratings:")
df.show(5)

## Step 4: Stages & Tasks (Narrow vs. Wide Transformations)

- **Narrow Transformations:** (e.g. `filter`, `select`) Data is processed locally inside each partition. Runs inside the **same Stage**.
- **Wide Transformations:** (e.g. `groupBy`, `join`) Requires data shuffling across executors. Splits execution into **multiple Stages**.
- **Tasks:** The number of Tasks launched within a Stage equals the number of partitions in that Stage.

### A. Narrow Transformation (Filter)
Filtering rating records is a narrow operation. Spark will pipeline this inside a single Stage.

In [ ]:
high_ratings = df.filter(df.rating >= 4.0)

# Action counts elements. Check YARN/Spark UI: this executes in 1 Stage.
print(f"High rating record count: {high_ratings.count()}")

### B. Wide Transformation (GroupBy & Average)
Aggregating scores per movie ID requires shuffling data by the group key `movieId` across partitions. This splits execution into 2 Stages (Map-side local grouping and Reduce-side global aggregation).

In [ ]:
avg_ratings = df.groupBy("movieId").agg(
    F.count("movieId").alias("count"),
    F.round(F.avg("rating"), 2).alias("avg_rating")
)

# Action displays elements. Check Spark UI: this splits into 2 Stages.
avg_ratings.show(5)

## Step 5: Stop the Spark Session

In [ ]:
spark.stop()
print("Spark Session stopped.")